# Revisión de matrices CompanyGame

Este notebook es la interfaz de auditoría de `cg_01`. Sirve para:

- ver qué fuentes y hojas existen;
- comprobar la fidelidad de los Markdown generados;
- comparar métricas entre periodos y fuentes compatibles;
- revisar anomalías, contradicciones e interpretaciones pendientes.

No sustituye la validación de los datos normalizados. Si algo no puede determinarse con evidencia, se muestra como advertencia o pregunta; no se inventa.

## 1. Preparación

La celda siguiente ejecuta el pipeline existente y regenera `outputs/f1/` y `outputs/markdown/`.
No modifica `data/`.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

from cg_01.excel_to_md import verify_literal_export
from cg_01.analysis import (
    anomalies_frame,
    compare_metric,
    load_interpretations,
    load_open_questions,
    markdown_fidelity_frame,
    open_questions_frame,
    period_pivot,
    raw_sheet_frame,
    run_review,
    sources_frame,
    source_overlap,
    sheet_index,
)

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)

review = run_review()
config = review.config
len(review.sources), len(review.observations), len(review.anomalies)

## 2. Fuentes disponibles

Cada fila es una fuente inventariada. `source_type` distingue Excel, JSON de decisiones y Markdown capturado manualmente. `kind` distingue, entre otras cosas, `generated_workbook` de `captured_markdown`.

In [ ]:
fuentes = sources_frame(review.sources)
display(fuentes)

## 3. Explorar una hoja cruda

Cambia `SOURCE_ID` y `SHEET` para inspeccionar otra fuente. La tabla muestra exactamente las celdas no vacías y su coordenada original.

In [ ]:
SOURCE_ID = review.sources[0].source_id
SHEET = review.sources[0].sheets[0]
print("Fuente:", SOURCE_ID)
print("Hojas:", review.sources[0].sheets)
display(raw_sheet_frame(review.cells_by_source, SOURCE_ID, SHEET).head(120))

## 4. Hojas exportadas y referencias de origen

F1.2 genera un `.md` por hoja con el contenido textual literal de cada celda no vacía. No interpreta tablas ni encabezados; el índice permite pasar de la celda a su coordenada original.

In [ ]:
hojas = sheet_index(review.markdown)
display(hojas.head(120))

if not hojas.empty:
    fila = hojas.iloc[0]
    ruta = str(Path(review.markdown["output_dir"]) / str(fila["md_file"]))
    print("Markdown de ejemplo:", ruta)

### Referencias de celdas

`cell_index.csv` permite pasar de una celda Markdown a la coordenada exacta del Excel. Filtra por `source`, `sheet` o `cell` para revisar la procedencia.

In [ ]:
cell_index = pd.read_csv(Path(review.markdown["output_dir"]) / "cell_index.csv")
display(cell_index.head(30))

if not hojas.empty:
    print("Ejemplo de vista previa del Markdown:\n")
    print(ruta)
    print(Path(ruta).read_text(encoding="utf-8")[:3500])

## 5. Fidelidad del Markdown

Comprobación clave: cada celda no vacía del Excel debe aparecer una sola vez en `cell_index.csv`. Un valor distinto de cero en `missing_cells` o `duplicate_cells` exige revisar el exportador antes de usar el Markdown.

In [ ]:
fidelidad = markdown_fidelity_frame(review.markdown)
display(fidelidad)

verificacion = verify_literal_export(
    Path(review.markdown["output_dir"]),
    review.sources,
)
print(
    "Índice:", verificacion["index_rows"],
    "| celdas perdidas:", len(verificacion["missing_cells"]),
    "| celdas extra:", len(verificacion["extra_cells"]),
    "| textos distintos:", len(verificacion["text_mismatches"]),
    "| duplicados:", len(verificacion["duplicates"]),
    "| filas Markdown ausentes:", len(verificacion["markdown_missing"]),
)

if not fidelidad.empty:
    assert (fidelidad["missing_cells"] == 0).all(), "Hay celdas no indexadas"
    assert (fidelidad["duplicate_cells"] == 0).all(), "Hay celdas duplicadas"
    assert not verificacion["missing_cells"], "Hay celdas esperadas que no están en el índice"
    assert not verificacion["extra_cells"], "Hay celdas en el índice que no existen en la fuente"
    assert not verificacion["text_mismatches"], "Hay textos distintos entre Excel e índice"
    assert not verificacion["markdown_missing"], "Hay celdas sin fila en el Markdown"
    print("Fidelidad verificada: coordenadas y valores coinciden con las celdas originales.")

## 6. Comparar métricas entre periodos

`compare_metric` usa la capa normalizada, no el Markdown. Busca por etiqueta normalizada y devuelve una fila por observación con su procedencia. Ajusta `METRIC` según lo que quieras revisar.

In [ ]:
METRIC = "resultado"
comparacion = compare_metric(review.observations, METRIC)
display(comparacion.head(60))

if not comparacion.empty:
    display(period_pivot(comparacion))

## 6.1. Ocupación: planta frente a línea

La penalización por baja ocupación se evalúa sobre la planta (`occupancy_penalty_mode`), mientras que el cálculo de costes industriales usa la ocupación por línea (`occupancy_cost_mode`). Esta celda muestra ambos niveles sin confundirlos.

In [ ]:
ocupacion = compare_metric(review.observations, "ocupacion")
display(ocupacion[["period", "source_id", "sheet", "label", "segment", "value", "unit"]])
print(
    "penalty_mode:", review.config.occupancy_penalty_mode,
    "| cost_mode:", review.config.occupancy_cost_mode,
)

## 6.2. Precedencia entre matrices y discrepancias

La política provisional prefiere la matriz posterior por ejercicio, métrica y periodo, sin borrar las versiones anteriores. Las diferencias se muestran, no se corrigen automáticamente.

In [ ]:
overlap = source_overlap(review.observations, "SRC-XLSX-002", "SRC-XLSX-003")
display(overlap)

if not overlap.empty:
    print("Discrepancias 2025 entre Matriz 1 y Matriz 2:", len(overlap))

## 7. Anomalías y contradicciones

Las anomalías se generan con las reglas de `validation.py`. Filtra por severidad o categoría para priorizar lo que afecta a balance, caja, deuda o resultado.

In [ ]:
anomalias = anomalies_frame(review.anomalies)
display(anomalias["severity"].value_counts())
display(anomalias["category"].value_counts())

prioritarias = anomalias[anomalias["severity"].isin(["critical", "high"])]
display(prioritarias[["anomaly_id", "severity", "rule_id", "source_id", "sheet", "cell", "period", "message"]].head(80))

## 8. Interpretaciones confirmadas y preguntas abiertas

El estado persistente vive en `.specify/memory/`. Una interpretación confirmada evita repetir la misma pregunta; una pregunta abierta debe incluir evidencia, alternativas y consecuencia. Este notebook solo muestra el estado: las respuestas se registran en esos archivos.

In [ ]:
interpretaciones = load_interpretations()
preguntas = load_open_questions()

display(Markdown("### Interpretaciones confirmadas"))
display(interpretations_frame := pd.DataFrame(interpretaciones.get("items", [])))

display(Markdown("### Preguntas abiertas"))
display(open_questions_frame(preguntas))

## 9. Limitaciones y decisiones aceptadas

- `Matriz_general_2` es el cierre real y ejecutado de 2027.
- El balance se calcula con el bloque detallado: reservas 930.663, residuales de 1 € documentados y D-02 como explicación del usuario sin documento independiente.
- Q-009 está aceptada como excepción: se conserva 30.000 real en bufandas 2027 y no se cambia la regla `0 = mantener`.
- INT-003 está aceptada como inferencia: valor bruto y normalizado se conservan separados.
- Siguen documentadas, sin bloquear F1.4, las diferencias de gastos generales, CAPEX, pasivo corriente trimestral, alias, pesos de valor y capturas.
- El notebook no implementa el modelo financiero, Monte Carlo ni optimización.